El notebook tiene como finalidad evaluar el procesamiento de diferentes volúmenes de datos sintéticos mediante Apache Spark en Databricks, utilizando información de ventas y detalles de ventas de una librería escolar.
se trabajara con 3 escenarios:
- BD01:10 000 VENTAS
- BD02:50 000 VENTAS
- BD03: 100 000 VENTAS
##En cada escenario se realizan operaciones de lectura, unión, transformación, filtrado y agregación para obtener indicadores comerciales y medir el tiempo de procesamiento.

**El punto de partida del proyecto está conformado por cuatro archivos SQL que permiten construir la base de datos transaccional, insertar los datos iniciales, crear el Data Mart y ejecutar el proceso de extcción, transformación y carga (ETL).ra**
A partir de esta estructura y de los datos iniciales de la librería, se prepararon datasets sintéticos de mayor volumen para evaluar el procesamiento de información mediante Apache Spark. Estos datasets fueron generados utilizando Google Colab y almacenados en formato Parquet para su posterior utilización en Databricks.

In [0]:
##CONFIGURACION DE LA RUTA DE ALMACENAMIENTO
#Define la ubicación principal donde se almacenan los archivos Parquet de los tres escenarios de prueba.
BASE_PATH = "/Volumes/solucion_bd/datos_bd/datos_libreria"

print("Ruta configurada:", BASE_PATH)

Ruta configurada: /Volumes/solucion_bd/datos_bd/datos_libreria


In [0]:
##VERIFICACION DE LOS ARCHIVOS ALMACENADOS
##listar los archivos y carpetas existentes en la ubicación configurada
display(
    dbutils.fs.ls(BASE_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/,BD01/,0,1795832325135
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD02/,BD02/,0,1795832325135
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD03/,BD03/,0,1795832325135


In [0]:
##Lee los archivos Parquet correspondientes a las ventas mediante spark.read.parquet
##prepara los conjuntos de datos para realizar operaciones de procesamiento distribuido con Spark.
ruta_ventas = f"{BASE_PATH}/BD01/venta_sintetica.parquet"
df_ventas = spark.read.parquet(ruta_ventas)

display(df_ventas.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado
108,4,11,4,2025-09-10 00:05:17,CANCELADO
109,10,4,3,2026-03-15 07:16:31,COMPLETADO
110,1,3,1,2026-07-13 16:12:26,COMPLETADO
111,9,3,2,2026-10-27 12:29:11,COMPLETADO
112,6,4,4,2026-11-09 13:21:44,COMPLETADO
113,10,12,4,2026-06-19 23:16:47,COMPLETADO
114,2,1,5,2026-09-15 08:38:51,COMPLETADO
115,9,12,5,2026-10-24 20:15:08,COMPLETADO
116,5,11,1,2026-05-02 05:43:16,PENDIENTE
117,4,2,3,2026-01-09 07:10:56,COMPLETADO


In [0]:
print("Total de ventas:", df_ventas.count())

Total de ventas: 10000


In [0]:
##Lee los archivos Parquet correspondientes a los detalles mediante spark.read.parquet
##prepara los conjuntos de datos para realizar operaciones de procesamiento distribuido con Spark.
ruta_detalles = f"{BASE_PATH}/BD01/ventaDetalle_sintetico.parquet"
df_detalles = spark.read.parquet(ruta_detalles)

display(df_detalles.limit(10))

print("Total de detalles:", df_detalles.count())

idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
183,108,7,5,17.0,12.0
184,108,9,5,28.0,18.0
185,109,3,1,1.5,0.8
186,109,21,4,13.0,7.5
187,109,2,1,1.5,0.8
188,110,7,1,17.0,12.0
189,110,22,4,7.5,4.0
190,110,16,3,10.5,6.0
191,111,21,5,13.0,7.5
192,111,11,2,55.0,35.0


Total de detalles: 29775


El método .count() calcula la cantidad de registros existentes en cada DataFrame.
## el primer caso se obtuvo 10 000 registros de ventas y 29 775 registros de detalles

In [0]:
#mostramos el total de registros por escenario
for escenario in ["BD01", "BD02", "BD03"]:

    ventas = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/venta_sintetica.parquet"
    )

    detalles = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/ventaDetalle_sintetico.parquet"
    )

    print(f"\n--- {escenario} ---")
    print("Ventas:", ventas.count())
    print("Detalles:", detalles.count())


--- BD01 ---
Ventas: 10000
Detalles: 29775

--- BD02 ---
Ventas: 50000
Detalles: 150069

--- BD03 ---
Ventas: 100000
Detalles: 300003


In [0]:
#confirma que se ha creado el directorio
display(
    dbutils.fs.ls(f"{BASE_PATH}/BD01/")
)

path,name,size,modificationTime
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/ventaDetalle_sintetico.parquet,ventaDetalle_sintetico.parquet,332399,1795811024000
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/venta_sintetica.parquet,venta_sintetica.parquet,191600,1795811024000


In [0]:
#muestra la cantidad de archivos y el tamaño de cada uno
for escenario in ["BD01", "BD02", "BD03"]:

    print(f"\nContenido de {escenario}:")

    archivos = dbutils.fs.ls(
        f"{BASE_PATH}/{escenario}/"
    )

    for archivo in archivos:
        print(archivo.name, archivo.size, "bytes")


Contenido de BD01:
ventaDetalle_sintetico.parquet 332399 bytes
venta_sintetica.parquet 191600 bytes

Contenido de BD02:
ventaDetalle_sintetico.parquet 1647852 bytes
venta_sintetica.parquet 954031 bytes

Contenido de BD03:
ventaDetalle_sintetico.parquet 3089090 bytes
venta_sintetica.parquet 1830132 bytes


In [0]:
df_ventas = spark.read.parquet(
    f"{BASE_PATH}/BD01/venta_sintetica.parquet"
)

df_detalles = spark.read.parquet(
    f"{BASE_PATH}/BD01/ventaDetalle_sintetico.parquet"
)

print("Ventas:", df_ventas.count())
print("Detalles:", df_detalles.count())

Ventas: 10000
Detalles: 29775


In [0]:
#indica la cantidad de registros por idVentaDetalle
display(df_ventas.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado
108,4,11,4,2025-09-10 00:05:17,CANCELADO
109,10,4,3,2026-03-15 07:16:31,COMPLETADO
110,1,3,1,2026-07-13 16:12:26,COMPLETADO
111,9,3,2,2026-10-27 12:29:11,COMPLETADO
112,6,4,4,2026-11-09 13:21:44,COMPLETADO
113,10,12,4,2026-06-19 23:16:47,COMPLETADO
114,2,1,5,2026-09-15 08:38:51,COMPLETADO
115,9,12,5,2026-10-24 20:15:08,COMPLETADO
116,5,11,1,2026-05-02 05:43:16,PENDIENTE
117,4,2,3,2026-01-09 07:10:56,COMPLETADO


In [0]:
#muestra la cantidad de detalles por idVentaDetalle
display(df_detalles.limit(10))

idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
183,108,7,5,17.0,12.0
184,108,9,5,28.0,18.0
185,109,3,1,1.5,0.8
186,109,21,4,13.0,7.5
187,109,2,1,1.5,0.8
188,110,7,1,17.0,12.0
189,110,22,4,7.5,4.0
190,110,16,3,10.5,6.0
191,111,21,5,13.0,7.5
192,111,11,2,55.0,35.0


ANALISIS Y PROCESAMIENTO

In [0]:
##El método .printSchema() muestra los nombres de las columnas y sus respectivos tipos de datos.
##muestra el esquema de los dataframes
print("COLUMNAS DE VENTAS")
df_ventas.printSchema()

print("\nCOLUMNAS DE DETALLES")
df_detalles.printSchema()

COLUMNAS DE VENTAS
root
 |-- idVenta: long (nullable = true)
 |-- fkCliente: long (nullable = true)
 |-- fkUsuario: long (nullable = true)
 |-- fkMetodoPago: long (nullable = true)
 |-- fecha: string (nullable = true)
 |-- estado: string (nullable = true)


COLUMNAS DE DETALLES
root
 |-- idVentaDetalle: long (nullable = true)
 |-- fkVenta: long (nullable = true)
 |-- fkProducto: long (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precioVenta: double (nullable = true)
 |-- costoUnitario: double (nullable = true)



In [0]:

print("Total de ventas:", df_ventas.count())

print("Total de detalles:", df_detalles.count())

Total de ventas: 10000
Total de detalles: 29775


In [0]:
#muestra la cantidad de ventas por estado
display(
    df_ventas.groupBy("estado")
    .count()
    .orderBy("count", ascending=False)
)

estado,count
COMPLETADO,8801
CANCELADO,736
PENDIENTE,463


In [0]:
#muestra la cantidad de detalles
display(
    df_detalles.groupBy()
    .count()
)

count
29775


PRUEBAS

In [0]:
#muestra la cantidad de ventas por estado
display(
    df_ventas
    .groupBy("estado")
    .count()
    .orderBy("estado")
)

estado,count
CANCELADO,736
COMPLETADO,8801
PENDIENTE,463


**Limpieza y preparación**

In [0]:
#carga de dataframes
ventas_dup = spark.read.parquet(f"{BASE_PATH}/BD01/venta_sintetica.parquet")
detalles_dup = spark.read.parquet(f"{BASE_PATH}/BD01/ventaDetalle_sintetico.parquet")

In [0]:
from pyspark.sql.functions import col, count

# Detectar duplicados en ventas
duplicados_ventas = (
    ventas_dup.groupBy("idVenta")
    .agg(count("*").alias("Repeticiones"))
    .filter(col("Repeticiones") > 1)
)

# Detectar duplicados en detalles
duplicados_detalles = (
    detalles_dup.groupBy("idVentaDetalle")
    .agg(count("*").alias("Repeticiones"))
    .filter(col("Repeticiones") > 1)
)

print("Duplicados en ventas:")
display(duplicados_ventas)

print("Duplicados en detalles:")
display(duplicados_detalles)

Duplicados en ventas:


idVenta,Repeticiones
624,2
849,2
2769,2
7816,2
1238,2
9613,2
2509,2
2910,2
4495,2
3662,2


Duplicados en detalles:


idVentaDetalle,Repeticiones
1636,2
588,2
29170,2
1226,2
1595,2
2185,2
4028,2
19518,2
5033,2
5081,2


In [0]:
# Aplicar limpieza
ventas_limpias = ventas_dup.dropDuplicates()
detalles_limpios = detalles_dup.dropDuplicates()

# Comparar registros
comparacion = [
    ("Ventas", ventas_dup.count(), ventas_limpias.count()),
    ("Detalles", detalles_dup.count(), detalles_limpios.count())
]

df_comparacion = spark.createDataFrame(
    comparacion,
    ["Tabla", "Antes_Limpieza", "Despues_Limpieza"]
)

display(df_comparacion)

Tabla,Antes_Limpieza,Despues_Limpieza
Ventas,10000,9805
Detalles,29775,29188


**Unión de tablas**

In [0]:
## Unión de ventas y detalles
#Realiza una unión entre los DataFrames de ventas y detalles utilizando el identificador de venta.
#Propósito: integrar la información de las ventas con sus respectivos productos y detalles para realizar el análisis comercial.

df_unificado = (
    ventas_limpias.alias("v")
    .join(
        detalles_limpios.alias("d"),
        col("v.idVenta") == col("d.fkVenta"),
        "inner"
    )
)

display(df_unificado.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado,idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,190,110,16,3,10.5,6.0
114,2,1,5,2026-09-15 08:38:51,COMPLETADO,200,114,18,1,9.0,5.0
115,9,12,5,2026-10-24 20:15:08,COMPLETADO,205,115,17,5,79.9,45.0
115,9,12,5,2026-10-24 20:15:08,COMPLETADO,207,115,4,1,1.2,0.6
122,1,1,1,2026-05-18 02:45:20,COMPLETADO,223,122,12,2,30.0,18.0
122,1,1,1,2026-05-18 02:45:20,COMPLETADO,224,122,16,3,10.5,6.0
123,5,12,2,2026-02-04 11:09:44,COMPLETADO,226,123,18,2,9.0,5.0
125,3,3,4,2025-07-11 00:42:44,COMPLETADO,233,125,12,1,30.0,18.0
136,2,11,2,2025-04-14 23:45:42,COMPLETADO,261,136,18,1,9.0,5.0
144,7,4,5,2026-07-25 21:09:56,COMPLETADO,286,144,6,5,2.0,1.0


In [0]:
##Transformación y cálculo de indicadores
#Calcula tres nuevos campos mediante operaciones aritméticas sobre las columnas existentes.
#Propósito: obtener medidas económicas que permitan analizar el comportamiento de las ventas y la utilidad generada.
from pyspark.sql.functions import col

df_analisis = (
    df_unificado
    .withColumn(
        "importeVenta",
        col("cantidad") * col("precioVenta")
    )
    .withColumn(
        "costoTotal",
        col("cantidad") * col("costoUnitario")
    )
    .withColumn(
        "utilidad",
        col("importeVenta") - col("costoTotal")
    )
)

display(df_analisis.limit(100))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado,idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario,importeVenta,costoTotal,utilidad
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,190,110,16,3,10.5,6.0,31.5,18.0,13.5
114,2,1,5,2026-09-15 08:38:51,COMPLETADO,200,114,18,1,9.0,5.0,9.0,5.0,4.0
115,9,12,5,2026-10-24 20:15:08,COMPLETADO,205,115,17,5,79.9,45.0,399.5,225.0,174.5
115,9,12,5,2026-10-24 20:15:08,COMPLETADO,207,115,4,1,1.2,0.6,1.2,0.6,0.6
122,1,1,1,2026-05-18 02:45:20,COMPLETADO,223,122,12,2,30.0,18.0,60.0,36.0,24.0
122,1,1,1,2026-05-18 02:45:20,COMPLETADO,224,122,16,3,10.5,6.0,31.5,18.0,13.5
123,5,12,2,2026-02-04 11:09:44,COMPLETADO,226,123,18,2,9.0,5.0,18.0,10.0,8.0
125,3,3,4,2025-07-11 00:42:44,COMPLETADO,233,125,12,1,30.0,18.0,30.0,18.0,12.0
136,2,11,2,2025-04-14 23:45:42,COMPLETADO,261,136,18,1,9.0,5.0,9.0,5.0,4.0
144,7,4,5,2026-07-25 21:09:56,COMPLETADO,286,144,6,5,2.0,1.0,10.0,5.0,5.0


In [0]:
#Este bloque pertenece a la etapa Gold del procesamiento Big Data. Su función es seleccionar exclusivamente las ventas completadas y calcular cuatro indicadores comerciales: ventas totales, costo total, utilidad total y unidades vendidas. Para ello se utilizan funciones de agregación de PySpark, cuyos resultados permiten evaluar el comportamiento comercial de la librería.
from pyspark.sql.functions import col, sum, round

df_completadas = df_analisis.filter(
    col("estado") == "COMPLETADO"
)

indicadores = df_completadas.agg(
    round(sum("importeVenta"), 2).alias("Ventas_Totales"),
    round(sum("costoTotal"), 2).alias("Costo_Total"),
    round(sum("utilidad"), 2).alias("Utilidad_Total"),
    sum("cantidad").alias("Unidades_Vendidas")
)

display(indicadores)

Ventas_Totales,Costo_Total,Utilidad_Total,Unidades_Vendidas
1404655.3,830252.6,574402.7,75315


In [0]:
## Análisis de las ventas por estado
#Agrupa los registros según el estado de la venta y calcula los importes acumulados.
#Propósito: observar el comportamiento económico de los diferentes estados de las transacciones.
from pyspark.sql.functions import col, sum, round

ventas_por_estado = (
    df_analisis
    .groupBy("estado")
    .agg(
        round(sum("importeVenta"), 2).alias("Ventas_Totales"),
        round(sum("costoTotal"), 2).alias("Costo_Total"),
        round(sum("utilidad"), 2).alias("Utilidad_Total"),
        sum("cantidad").alias("Unidades_Vendidas")
    )
    .orderBy("estado")
)

display(ventas_por_estado)

estado,Ventas_Totales,Costo_Total,Utilidad_Total,Unidades_Vendidas
CANCELADO,111175.4,65440.8,45734.6,6341
COMPLETADO,1404655.3,830252.6,574402.7,75315
PENDIENTE,69604.5,41200.3,28404.2,3982


Databricks visualization. Run in Databricks to view.

calcular indicadores

In [0]:
##Filtrado de ventas completadas y cálculo de indicadores
#Selecciona exclusivamente las ventas con estado COMPLETADO y calcula los indicadores comerciales.
#Propósito: obtener indicadores comerciales válidos para el análisis, excluyendo las transacciones pendientes y canceladas.
from pyspark.sql.functions import col, sum, countDistinct, round

indicadores_BD01 = df_analisis.filter(
    col("estado") == "COMPLETADO"
).agg(
    countDistinct("idVenta").alias("Total_Ventas"),
    sum("cantidad").alias("Unidades_Vendidas"),
    round(sum("importeVenta"), 2).alias("Ventas_Totales"),
    round(sum("costoTotal"), 2).alias("Costo_Total"),
    round(sum("utilidad"), 2).alias("Utilidad_Total")
)

display(indicadores_BD01)

Total_Ventas,Unidades_Vendidas,Ventas_Totales,Costo_Total,Utilidad_Total
8602,75315,1404655.3,830252.6,574402.7


In [0]:
import time

In [0]:
##Comparación de los escenarios BD01, BD02 y BD03
#import time: permite medir el tiempo de ejecución.
#import math: proporciona funciones matemáticas.
#resultados = []: crea una lista para almacenar los resultados.
#for escenario in [...]: recorre automáticamente los tres escenarios.

#El tiempo de procesamiento se mide mediante la función time.time() de Python. La medición comienza antes de realizar la limpieza de los DataFrames y termina después de ejecutar las operaciones de unión, transformación y cálculo de indicadores.

#Propósito: ejecutar el mismo procedimiento de análisis sobre diferentes volúmenes de información.

import time
import math
from pyspark.sql.functions import col, sum, countDistinct, round

resultados = []

for escenario in ["BD01", "BD02", "BD03"]:

    # Leer archivos reemplazados por los datos duplicados
    ventas_dup = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/venta_sintetica.parquet"
    )

    detalles_dup = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/ventaDetalle_sintetico.parquet"
    )

    # Inicio de la medición
    inicio = time.time()

    # Limpiar registros duplicados exactos
    ventas = ventas_dup.dropDuplicates()
    detalles = detalles_dup.dropDuplicates()

    # Unir tablas limpias
    unificado = ventas.alias("v").join(
        detalles.alias("d"),
        col("v.idVenta") == col("d.fkVenta"),
        "inner"
    )

    # Calcular importes y utilidad
    analisis = (
        unificado
        .withColumn(
            "importeVenta",
            col("cantidad") * col("precioVenta")
        )
        .withColumn(
            "costoTotal",
            col("cantidad") * col("costoUnitario")
        )
        .withColumn(
            "utilidad",
            col("importeVenta") - col("costoTotal")
        )
    )

    # Filtrar ventas completadas
    completadas = analisis.filter(
        col("estado") == "COMPLETADO"
    )

    # Calcular indicadores comerciales
    indicadores = completadas.agg(
        countDistinct("idVenta").alias("Total_Ventas"),
        sum("cantidad").alias("Unidades_Vendidas"),
        round(sum("importeVenta"), 2).alias("Importe_Total"),
        round(sum("costoTotal"), 2).alias("Costo_Total"),
        round(sum("utilidad"), 2).alias("Utilidad_Total")
    ).first()

    total_ventas = indicadores["Total_Ventas"] or 0
    total_unidades = indicadores["Unidades_Vendidas"] or 0
    total_importe = indicadores["Importe_Total"] or 0
    total_costo = indicadores["Costo_Total"] or 0
    total_utilidad = indicadores["Utilidad_Total"] or 0

    # Contar registros después de la limpieza
    cantidad_ventas = ventas.count()
    cantidad_detalles = detalles.count()

    # Fin de la medición
    fin = time.time()
    tiempo = math.floor((fin - inicio) * 10000) / 10000

    # Guardar resultados
    resultados.append((
        escenario,
        cantidad_ventas,
        cantidad_detalles,
        total_ventas,
        total_unidades,
        float(total_importe),
        float(total_costo),
        float(total_utilidad),
        tiempo
    ))

# Crear tabla comparativa
columnas = [
    "Escenario",
    "Ventas_Limpias",
    "Detalles_Limpios",
    "Ventas_Completadas",
    "Unidades_Vendidas",
    "Importe_Total",
    "Costo_Total",
    "Utilidad_Total",
    "Tiempo_Procesamiento"
]

df_resultados = spark.createDataFrame(resultados, columnas)

display(df_resultados)

Escenario,Ventas_Limpias,Detalles_Limpios,Ventas_Completadas,Unidades_Vendidas,Importe_Total,Costo_Total,Utilidad_Total,Tiempo_Procesamiento
BD01,9805,29188,8602,75315,1404655.3,830252.6,574402.7,1.8208
BD02,49015,147123,43028,381148,7043626.7,4160732.4,2882894.3,2.0118
BD03,98032,294130,85978,763100,1.41184141E7,8344616.1,5773798.0,2.35


Databricks visualization. Run in Databricks to view.

**“Análisis de ventas mensuales”**

In [0]:
## Análisis de ventas mensuales
# Agrupa las ventas completadas por mes y calcula los importes y unidades vendidas.
# Propósito: identificar el comportamiento de las ventas a lo largo del tiempo.
# Se muestra los mejores 5 meses de ventas totales
from pyspark.sql.functions import col, sum, to_date, date_format, round

ventas_mensuales = (
    df_analisis
    .filter(col("estado") == "COMPLETADO")
    .withColumn("fecha_convertida", to_date(col("fecha")))
    .withColumn("Mes", date_format(col("fecha_convertida"), "yyyy-MM"))
    .groupBy("Mes")
    .agg(
        round(sum("importeVenta"), 2).alias("Ventas_Totales"),
        sum("cantidad").alias("Unidades_Vendidas")
    )
    .orderBy(col("Ventas_Totales").desc())
    .limit(5)
)

display(ventas_mensuales)


Mes,Ventas_Totales,Unidades_Vendidas
2025-12,71039.5,3632
2026-03,68968.2,3736
2025-05,68486.8,3744
2025-08,66006.5,3535
2025-07,65130.1,3378


Databricks visualization. Run in Databricks to view.